# Train the beam GNN (simple)
Runtime > Change runtime type > **T4 GPU**. Then run the cells one by one, top to bottom (Shift+Enter).

**1. Connect Google Drive**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**2. Get the latest code from GitHub** (uses your GH_TOKEN secret)

In [ ]:
import os, subprocess, sys
from google.colab import userdata
token = userdata.get("GH_TOKEN")
if os.path.exists("/content/GNN-BeamRebarDetector"):
    subprocess.run(["git", "-C", "/content/GNN-BeamRebarDetector", "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://{token}@github.com/yeongit5985/GNN-BeamRebarDetector.git", "/content/GNN-BeamRebarDetector"])
sys.path.insert(0, "/content/GNN-BeamRebarDetector")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdfium2", "rapidocr_onnxruntime"])
print("code ready")

**3. Your folders** (`G:\\My Drive\\Deep Learning\\GNN-BeamRebarDetector`)

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta


BASE  = Path("/content/drive/MyDrive/Deep Learning/GNN-BeamRebarDetector")      # your working directory
JSONS = BASE / "FinalJson"      # your checked pages
PDFS  = BASE / "PDFDwg"         # the drawings

my_tz = timezone(timedelta(hours=8))    # Set Malaysia timezone (UTC+8)

OUT   = BASE / "gnn_training" / datetime.now(my_tz).strftime("%y%m%d_%H%M")   # graphs, model and charts go here
OUT.mkdir(parents=True, exist_ok=True)

pages = []
for f in sorted(JSONS.glob("*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    pages += d if isinstance(d, list) else [d]
(OUT / "all_pages.json").write_text(json.dumps(pages), encoding="utf-8")
print(len(pages), "pages found")

**4. Turn each page into a graph** (a few minutes)

In [ ]:
import graph_builder as gb
gb.export_to_graphs(OUT / "all_pages.json", PDFS, OUT / "graphs")

**5. Train** on your real pages. The pages in `TEST_SHEETS` are kept aside to test the model on drawings it has not seen; they stay the same every run, so runs can be compared. Stops early when the test score stops improving; the best version is kept.

In [ ]:
import re
import gnn

graphs = gnn.load_graphs(OUT / "graphs", verbose=False)

# The same test pages every run, so a change in the scores comes from the code, not from other pages
TEST_SHEETS = {8, 39, 44}
# Revisions of one drawing stay on the same side (all train or all test): otherwise the test page is seen in training
SAME_DRAWING = [{3, 20}]

def sheet_no(g):                                # '261002_PDFDwg 8_p0' -> 8
    m = re.search(r"PDFDwg\s*(\d+)", g["name"])
    return int(m.group(1)) if m else None

for group in SAME_DRAWING:
    if 0 < len(group & TEST_SHEETS) < len(group):
        raise ValueError(f"sheets {sorted(group)} are one drawing: put all of them in TEST_SHEETS, or none")
test_pages = [g for g in graphs if sheet_no(g) in TEST_SHEETS]
train_pages = [g for g in graphs if sheet_no(g) not in TEST_SHEETS]
missing = TEST_SHEETS - {sheet_no(g) for g in test_pages}
if missing:
    print("WARNING: test sheets not in the graphs:", sorted(missing))

print(f"train: {len(train_pages)} pages | test: {len(test_pages)} pages, sheets {sorted(sheet_no(g) for g in test_pages)}")

state = {"best": -1, "bad": 0}
def watch(row):                                # early stopping: 5 checks (50 epochs) without improvement
    if "node_f1" in row:
        score = (row["node_f1"] + row["relation_ap"]) / 2
        if score > state["best"]:
            state["best"], state["bad"] = score, 0
        else:
            state["bad"] += 1

model, history, scores = gnn.train(train_pages, test_pages, epochs=200,
                                   progress=watch, should_stop=lambda: state["bad"] >= 5)
print("kept the model of epoch", scores["epoch"])

**6. Tested pages** as review pages (open `tested pages/review_index.html`)


In [ ]:
import subprocess
import review_relations as rr
from pdf_backend import pymupdf

# Tested pages as review pages (review_*.html), one per test page, in the 'tested pages' sub-folder
test_dir = OUT / "tested pages"
img_dir = test_dir / "img"
img_dir.mkdir(parents=True, exist_ok=True)

test_names = {g["name"] for g in test_pages}
test_tasks = []
for t in pages:
    where = gb.pdf_page_for_task(t, PDFS)
    if not where or f"{Path(where[0]).stem}_p{where[1]}" not in test_names:
        continue
    test_tasks.append(t)

    # The page image the review page shows, named as in the annotation (rendered from the PDF at 150 dpi)
    img_name = rr.UPLOAD_PREFIX.sub("", rr.image_name(str(t["data"]["image"])))
    page = pymupdf.open(where[0])[where[1]]
    if getattr(page, "rotation", 0):
        page.remove_rotation()
    page.get_pixmap(dpi=150, colorspace=pymupdf.csRGB).save(img_dir / (Path(img_name).stem + ".png"))

(test_dir / "test_pages.json").write_text(json.dumps(test_tasks), encoding="utf-8")
subprocess.run([sys.executable, "/content/GNN-BeamRebarDetector/review_relations.py", str(test_dir / "test_pages.json"),
                "--images", str(img_dir), "--pdf-dir", str(PDFS), "--out", str(test_dir)], check=True)
print(f"Saved {len(test_tasks)} of {len(test_pages)} test pages as review pages to {test_dir}")

**7. Scores** (F1: 1.00 = perfect)

In [ ]:
print(gnn.metrics_text(scores))

**8. Training curves** (saved as training_curve.png)

In [ ]:
import matplotlib.pyplot as plt
ep_all  = [h["epoch"] for h in history]
checked = [h for h in history if "node_f1" in h]
ep_chk  = [h["epoch"] for h in checked]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ep_all, [h["loss"] for h in history], color="#2a6fdb", linewidth=2)
ax[0].set_title("Training loss (lower = learning)"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
ax[1].plot(ep_chk, [h["node_f1"] for h in checked], "-o", color="#2a6fdb", linewidth=2, label="Labels (what each item is)")
ax[1].plot(ep_chk, [h["relation_ap"] for h in checked], "-o", color="#e8833a", linewidth=2, label="Links (what connects to what)")
ax[1].axvline(scores["epoch"], color="gray", linestyle="--", label=f"Model kept (epoch {scores['epoch']})")
ax[1].set_ylim(0, 1); ax[1].set_title(f"Score on the {len(test_pages)} test pages (higher = better)")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Score (1.0 = perfect)"); ax[1].legend(loc="lower right")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "training_curve.png", dpi=150); plt.show()

**9. Score per type** (saved as scores_per_type.png)

In [ ]:
pc, pk = scores["per_class"], scores["per_kind"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(list(pc), [v["f1"] for v in pc.values()], color="#2a6fdb"); ax[0].set_title("Labels: score per type")
ax[1].barh(list(pk), [v["f1"] for v in pk.values()], color="#e8833a"); ax[1].set_title("Links: score per type")
for a in ax:
    a.set_xlim(0, 1); a.invert_yaxis(); a.grid(axis="x", alpha=0.3)
    for bar in a.patches:
        a.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.2f}", va="center")
plt.tight_layout(); plt.savefig(OUT / "scores_per_type.png", dpi=150); plt.show()

**10. Overfitting check**: a big gap between training and test pages = memorising (saved with the scores in step 11)

In [ ]:
train_score = gnn.evaluate(model, train_pages, model.thresholds)
print(f"Training pages: labels {train_score['node_f1']:.2f} | links {train_score['relation_f1']:.2f}")
print(f"Test pages:     labels {scores['node_f1']:.2f} | links {scores['relation_f1']:.2f}")
# kept in beam_gnn.metrics.json (step 11), so runs can be compared later
scores["train_pages_check"] = {k: train_score[k] for k in ("node_f1", "relation_f1", "relation_ap")}
scores["train_pages_check"]["per_kind_f1"] = {k: v["f1"] for k, v in train_score["per_kind"].items()}

**11. Save the model**

In [ ]:
path = gnn.save_model(model, OUT / "model" / "beam_gnn.pt", metrics=scores,
                      trained_on=[g["name"] for g in train_pages], example=graphs[0])
print("saved:", path)